<a href="https://colab.research.google.com/github/harshvardhaniimi/isa383-google-colab/blob/main/09_Pandas_Basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pandas for Data Wrangling (Basic) 🚢

### ISA 383: Python for Business Analytics

This is the first of two notebooks on pandas, Python's main library for working with tables.
Our dataset is one of the most famous in data science: the passenger list of the RMS *Titanic*.
You will learn how to build, read, inspect, reshape, select, filter, and sort tables while answering questions that two official inquiries asked in 1912.

## Using this notebook in Google Colab

1. Before editing, select **File > Save a copy in Drive**.
2. Run the cells in order. Some practice cells intentionally wait for your input.
3. The notebook creates any course folders and teaching files it needs automatically.
4. Files under `/content` are temporary and disappear when the Colab runtime resets.
5. Never paste an API key into a notebook cell. Use the **Secrets** panel when instructed.

You do not need to find, copy, or type a file path for the prepared course data.

## Learning objectives 🎯

By the end of this notebook, you should be able to:

1. Create and inspect `Series` and `DataFrame` objects, and explain how the index labels rows.
2. Read CSV files and Excel workbooks (including multiple sheets) with useful parameters, and write results back to disk.
3. Select columns and rows with `[]`, `loc`, and `iloc`, filter rows with Boolean conditions, and sort with `sort_values()`.
4. State the three tidy data principles and identify where a table breaks them.
5. Convert between wide and long layouts with `melt()` and `pivot()`, and explain when each layout is useful.

## 1. The story behind the data 🧊

On 10 April 1912, the RMS *Titanic* left Southampton, England, on its maiden voyage to New York.
It was operated by the White Star Line, built by Harland & Wolff in Belfast, and was the largest ship afloat.
A 1911 trade magazine had described its design as "practically unsinkable."
After calling at Cherbourg in France and Queenstown (now Cobh) in Ireland, the ship headed west across the Atlantic with about 2,224 passengers and crew on board.

At about 11:40 p.m. on 14 April, the *Titanic* struck an iceberg south of Newfoundland.
By about 2:20 a.m. on 15 April, less than three hours later, it had sunk.
The ship carried 20 lifeboats with room for about 1,178 people, roughly half of those on board.
More than 1,500 people died in the freezing water.
The RMS *Carpathia* reached the lifeboats at dawn and rescued about 700 survivors.

| Date (1912) | Event |
|---|---|
| 2 April | Leaves the Belfast shipyard for Southampton (no paying passengers) |
| 10 April, noon | Maiden voyage begins at Southampton, England |
| 10 April, evening | Calls at Cherbourg, France |
| 11 April | Calls at Queenstown (Cobh), Ireland, then sets out across the Atlantic |
| 14 April, about 11:40 p.m. | Strikes an iceberg |
| 15 April, about 2:20 a.m. | Sinks |
| 15 April, morning | RMS *Carpathia* picks up the survivors from the lifeboats |

### Why analysts still study the *Titanic* 🔍

Within days of the sinking, the United States Senate opened an inquiry, and the British Wreck Commissioner followed soon after.
Their central questions were data questions.
Who got a place in the lifeboats?
Was the custom of "women and children first" actually followed?
Did a passenger's ticket class affect their chance of survival?

The disaster also led to the first International Convention for the Safety of Life at Sea (SOLAS) in 1914, which required lifeboat space for everyone on board.
More than a century later, the passenger list remains one of the most widely used teaching datasets because it is small, real, and full of human stories that numbers can help explain.

### Meet the data 📋

Our main file is `data/titanic.csv`, the 891-passenger training set from Kaggle's *Titanic: Machine Learning from Disaster* competition.
It is a subset of a passenger list compiled from *Encyclopedia Titanica* records.
It does **not** contain every passenger (there were roughly 1,300), and it contains **no crew members**.
Keep that limitation in mind: every conclusion we draw describes these 891 passengers.

| Column | Meaning | Example |
|---|---|---|
| `PassengerId` | Row identifier added by the dataset's creators | `1` |
| `Survived` | `1` = survived, `0` = died | `0` |
| `Pclass` | Ticket class: `1` = First, `2` = Second, `3` = Third | `3` |
| `Name` | Surname, title, and given names | `Braund, Mr. Owen Harris` |
| `Sex` | `male` or `female` | `male` |
| `Age` | Age in years (fractions for infants) | `22.0` |
| `SibSp` | Number of siblings or spouses aboard | `1` |
| `Parch` | Number of parents or children aboard | `0` |
| `Ticket` | Ticket number | `A/5 21171` |
| `Fare` | Ticket price in 1912 British pounds (£) | `7.25` |
| `Cabin` | Cabin number, if recorded | `C85` |
| `Embarked` | Port of embarkation: `S` = Southampton, `C` = Cherbourg, `Q` = Queenstown | `S` |

We also use `data/titanic_extra.xlsx`, an Excel workbook with two more tables.
The `ports` sheet describes the ports the ship called at.
The `board_of_trade` sheet is the official 1912 count of everyone aboard, **including the crew**, broken down by class, age group, sex, and outcome.
(The official count of 2,201 people differs slightly from later estimates such as 2,224; historians still debate the exact number.)

### 🕵️ Your role: analyst for the inquiry

Imagine you have been hired by the 1912 inquiry to answer its questions with data.
This notebook covers the first five questions; the Advanced notebook finishes the investigation.

| # | Inquiry question | pandas skill | Notebook |
|---|---|---|---|
| Q1 | Who was on board: how many passengers, in which classes, from which ports? | reading, inspecting, `value_counts()` | Basic |
| Q2 | What share of passengers survived? | Series methods | Basic |
| Q3 | Was "women and children first" followed? | filtering | Basic |
| Q4 | Who paid the most, and who travelled for free? | sorting | Basic |
| Q5 | How do the 891 passengers compare with everyone aboard, including crew? | tidy data, reshaping | Basic |
| Q6 | Did first class buy a better chance, and did that depend on sex and age? | `groupby()`, pivot tables | Advanced |
| Q7 | Did the port of embarkation matter, or was it really ticket class? | `merge()`, `groupby()` | Advanced |

Look for the 🕵️ symbol: it marks a cell that answers one of these questions.

## 2. Setup 🧰

We import pandas with its conventional alias `pd`.
The setup cell also downloads the course data files from the course GitHub repository into a temporary `data` folder.
Colab already includes `openpyxl`, the package pandas uses to read Excel files.

In [3]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Show more columns before pandas abbreviates the display with "...".
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

from urllib.request import urlretrieve

DATA_DIR = Path("/content/isa383/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download the course data from the course GitHub repository (only the first time).
DATA_URL = "https://raw.githubusercontent.com/harshvardhaniimi/isa383-google-colab/main/data/"
for filename in ["titanic.csv", "titanic_extra.xlsx"]:
    if not (DATA_DIR / filename).exists():
        urlretrieve(DATA_URL + filename, DATA_DIR / filename)
print("Course data is ready.")
print("pandas version:", pd.__version__)

Course data is ready.
pandas version: 2.2.3


## 3. Series: one labeled column 🧱

A `Series` is a one-dimensional sequence of values with an **index** of labels.
You can think of it as a cross between a list (ordered values) and a dictionary (labels that point to values).
Every Series also has a `dtype`, which tells pandas what kind of data it stores, and an optional `name`.
Operations apply to every value at once, a property called *vectorization*, so you rarely need a `for` loop to transform a Series.

### 🧭 Syntax first: Creating and using a Series

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
pd.Series(data, index=None, name=None, dtype=None)

series.index            # the labels
series.dtype            # the data type
series["label"]         # select a value by label
series.iloc[0]          # select a value by position
series.mean(), series.max(), series.idxmax(), series.describe()
```

Documentation: [Series](https://pandas.pydata.org/docs/reference/api/pandas.Series.html) · [Series.idxmax](https://pandas.pydata.org/docs/reference/api/pandas.Series.idxmax.html)

In [4]:
# Ages of the first five passengers in the dataset, created from a list.
ages = pd.Series([22, 38, 26, 35, 35], name="age")
ages

,age
0,22
1,38
2,26
3,35
4,35


The numbers on the left (0 to 4) are the default index.
The last line shows the Series name and its dtype, `int64` (64-bit integers).

In [5]:
# A dictionary gives each value a meaningful label: here, the passenger's surname.
fares = pd.Series(
    {"Braund": 7.25, "Cumings": 71.28, "Heikkinen": 7.93, "Futrelle": 53.10, "Allen": 8.05},
    name="fare_gbp",
)

print("Fare paid by Cumings:", fares["Cumings"])    # select by label
print("First fare in the Series:", fares.iloc[0])   # select by position
fares

Fare paid by Cumings: 71.28
First fare in the Series: 7.25


,fare_gbp
Braund,7.25
Cumings,71.28
Heikkinen,7.93
Futrelle,53.10
Allen,8.05


In [6]:
l = [0, 1, 2, 3]
l[2]

2

In [7]:
fares.iloc[4]

np.float64(8.05)

In [8]:
fares.loc["Allen"]

np.float64(8.05)

In [9]:
fares[2]

/tmp/ipykernel_4771/3272062191.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  fares[2]


np.float64(7.93)

In [10]:
l * 3

[0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3]

In [11]:
# Before 1971, one British pound was worth 20 shillings.
# A single expression converts every fare at once: no loop needed.
fares_in_shillings = fares * 20
fares_in_shillings.round(1)

,fare_gbp
Braund,145.0
Cumings,1425.6
Heikkinen,158.6
Futrelle,1062.0
Allen,161.0


In [12]:
print("Average fare: £", round(fares.mean(), 2))
print("Highest fare: £", fares.max(), "paid by", fares.idxmax())
print()

# A comparison returns a Boolean Series with one True/False per label.
fares > 50

Average fare: £ 29.52
Highest fare: £ 71.28 paid by Cumings



,fare_gbp
Braund,False
Cumings,True
Heikkinen,False
Futrelle,True
Allen,False


In [13]:
fares>fares.mean()

,fare_gbp
Braund,False
Cumings,True
Heikkinen,False
Futrelle,True
Allen,False


In [14]:
fares.loc[fares>fares.mean()]

,fare_gbp
Cumings,71.28
Futrelle,53.10


In [15]:
# pandas lines values up by LABEL, not by position.
cabin_upgrades = pd.Series({"Futrelle": 10.0, "Braund": 2.0})
print(fares)
print(cabin_upgrades)
fares + cabin_upgrades

Braund        7.25
Cumings      71.28
Heikkinen     7.93
Futrelle     53.10
Allen         8.05
Name: fare_gbp, dtype: float64
Futrelle    10.0
Braund       2.0
dtype: float64


,0
Allen,NaN
Braund,9.25
Cumings,NaN
Futrelle,63.10
Heikkinen,NaN


Only Braund and Futrelle appear in both Series, so only they receive a number.
The other three become `NaN` ("not a number"), the marker pandas uses for a missing value.
Index alignment is one of the most common ways missing values quietly enter an analysis.

> 💭 **Quick thought question:** Why might automatic alignment by label be safer than alignment by position when two lists of passengers are sorted differently?
When could it surprise you?

### 🧩 Guided practice: Series basics

Of the five passengers above, Braund and Allen died, while Cumings, Heikkinen, and Futrelle survived.
Create a Series called `survived` with surnames as the index, `1` for survived, and `0` for died.
Then print the share of these five passengers who survived.

*Hint:* the mean of a column of 0s and 1s is the proportion of 1s.

In [16]:
# TODO: Create the survived Series with surnames as the index.
survived = pd.Series(
    # your dictionary here
    {"Braund": 0, "Allen": 0, "Cumings": 1, "Heikkinen": 1, "Futrelle": 1},
    name = "survived"
)
survived

# TODO: Print the share of these five passengers who survived.
print(survived.mean())
print(survived.sum()/len(survived))

0.6
0.6


## 4. DataFrame: a table of aligned Series 📋

A `DataFrame` is a two-dimensional table.
Each column is a Series, and all columns share the same row index.
A column holds a single dtype, but different columns can hold different dtypes, which is why one row can mix text and numbers.

You can build a DataFrame from a **dictionary of lists** (one list per column) or from a **list of dictionaries** (one dictionary per row).
The second form is exactly the shape of JSON records returned by most web APIs, which you will meet later in the course.

### 🧭 Syntax first: Creating and describing a DataFrame

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
pd.DataFrame(data, index=None, columns=None)

df.shape        # (number of rows, number of columns)
df.columns      # column labels
df.index        # row labels
df.dtypes       # one dtype per column
df["column"]    # one column, returned as a Series
```

Documentation: [DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) · [DataFrame.dtypes](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dtypes.html)

In [17]:
# Dictionary of lists: each key becomes a column.
passengers = pd.DataFrame(
    {
        "name": ["Braund", "Cumings", "Heikkinen", "Futrelle", "Allen"],
        "pclass": [3, 2, 3, 1, 3],
        "sex": ["male", "female", "female", "female", "male"],
        "age": [22, 38, 26, 35, 35],
        "fare": [7.25, 71.28, 7.93, 53.10, 8.05],
        "survived": [0, 1, 1, 1, 0],
    }
)
passengers

,name,pclass,sex,age,fare,survived
0,Braund,3,male,22,7.25,0
1,Cumings,2,female,38,71.28,1
2,Heikkinen,3,female,26,7.93,1
3,Futrelle,1,female,35,53.10,1
4,Allen,3,male,35,8.05,0


In [18]:
survived.tolist()

[0, 0, 1, 1, 1]

In [19]:
passengers.columns.tolist()[1:3]

['pclass', 'sex']

In [20]:
print("Shape:", passengers.shape)
print("Columns:", passengers.columns.tolist())
print("Index:", passengers.index.tolist())
print()
print(passengers.dtypes)

Shape: (5, 6)
Columns: ['name', 'pclass', 'sex', 'age', 'fare', 'survived']
Index: [0, 1, 2, 3, 4]

name         object
pclass        int64
sex          object
age           int64
fare        float64
survived      int64
dtype: object


In [21]:
# List of dictionaries: each dictionary is one row, like a JSON record from an API.
records = [
    {"name": "Moran", "pclass": 3, "sex": "male", "age": None, "fare": 8.46, "survived": 0},
    {"name": "McCarthy", "pclass": 1, "sex": "male", "age": 54, "fare": 51.86, "survived": 0},
]
more_passengers = pd.DataFrame(records)
more_passengers

,name,pclass,sex,age,fare,survived
0,Moran,3,male,NaN,8.46,0
1,McCarthy,1,male,54.0,51.86,0


In [22]:
more_passengers.dtypes

,0
name,object
pclass,int64
sex,object
age,float64
fare,float64
survived,int64


In [23]:
more_passengers["age"]

,age
0,NaN
1,54.0


Mr. Moran's age was not recorded, so Python's `None` became `NaN`, and the `age` column became a float column because `NaN` is technically a float.

In [24]:
# One column of a DataFrame is a Series, so every Series method works on it.
fare_column = passengers["fare"]
print(type(fare_column))
print("Average fare of the five passengers: £", round(fare_column.mean(), 2))

<class 'pandas.core.series.Series'>
Average fare of the five passengers: £ 29.52


> 💭 **Quick thought question:** In `passengers`, what does one row represent, and what does one column represent?
Hold on to your answer: it is the heart of tidy data in Section 10.

### 🧩 Guided practice: Build a DataFrame

Our passenger file has no crew, so let's record four well-known crew members by hand.

| name | role | age | survived |
|---|---|---|---|
| Edward Smith | Captain | 62 | 0 |
| William Murdoch | First Officer | 39 | 0 |
| Jack Phillips | Senior wireless operator | 25 | 0 |
| Harold Bride | Junior wireless operator | 22 | 1 |

Create a DataFrame called `crew` from this table, using whichever construction style you prefer.
Print its shape and dtypes, and compute the average age.

In [25]:
# TODO: Create the crew DataFrame.
crew = {
    "name": ["Edward Smith", "William Murdoch", "Jack Philips", "Harold Bride"],
    "role": ["Captain", "First Officer", "Senior wireless operator", "Junior wireless operator"],
    "age": [62, 39, 25, 22],
    "survived": [0,0,0,1]
}
df_crew = pd.DataFrame(crew)
print(df_crew)
display(df_crew)
# TODO: Print crew.shape, crew.dtypes, and the average age.
print(f"{df_crew.shape=}, \n{df_crew.dtypes}, \n{df_crew['age'].mean()}")

              name                      role  age  survived
0     Edward Smith                   Captain   62         0
1  William Murdoch             First Officer   39         0
2     Jack Philips  Senior wireless operator   25         0
3     Harold Bride  Junior wireless operator   22         1


,name,role,age,survived
0,Edward Smith,Captain,62,0
1,William Murdoch,First Officer,39,0
2,Jack Philips,Senior wireless operator,25,0
3,Harold Bride,Junior wireless operator,22,1


df_crew.shape=(4, 4), 
name        object
role        object
age          int64
survived     int64
dtype: object, 
37.0


## 5. Reading CSV and Excel files 💾

In practice, data arrives in a file rather than being typed by hand.
`pd.read_csv()` reads comma-separated text files, and `pd.read_excel()` reads Excel workbooks.
Both return a DataFrame, and both accept parameters that save work later: which columns to keep, which column becomes the index, how many rows to read, and which strings mean "missing."
Always inspect what you read; a file that loads without an error can still be wrong.

### 🧭 Syntax first: Reading and writing files

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
pd.read_csv(
    filepath,
    usecols=None,      # list of columns to keep
    index_col=None,    # column to use as the row labels
    nrows=None,        # read only the first n rows
    na_values=None,    # extra strings that should mean "missing"
    dtype=None,        # force column types, e.g. {"Pclass": "category"}
)

pd.read_excel(filepath, sheet_name=0)      # one sheet, by position or by name
pd.read_excel(filepath, sheet_name=None)   # every sheet, as a dict of DataFrames

df.to_csv(filepath, index=False)
df.to_excel(filepath, sheet_name="Sheet1", index=False)
```

Documentation: [read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [read_excel](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html) · [to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html) · [to_excel](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_excel.html)

In [26]:
titanic = pd.read_csv(DATA_DIR / "titanic.csv")
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [27]:
# Keep only four columns and the first five rows: handy when a file is large.
pd.read_csv(
    DATA_DIR / "titanic.csv",
    usecols=["Name", "Sex", "Age", "Survived"],
    nrows=5,
)

,Survived,Name,Sex,Age
0,0,"Braund, Mr. Owen Harris",male,22.0
1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0
2,1,"Heikkinen, Miss. Laina",female,26.0
3,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0
4,0,"Allen, Mr. William Henry",male,35.0


In [28]:
# Use PassengerId as the row labels instead of 0, 1, 2, ...
titanic_by_id = pd.read_csv(DATA_DIR / "titanic.csv", index_col="PassengerId")
titanic_by_id.head(3)

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


**Excel workbooks** can contain several sheets.
With `sheet_name=None`, pandas reads every sheet and returns a dictionary whose keys are the sheet names and whose values are DataFrames.

In [29]:
excel_file = DATA_DIR / "titanic_extra.xlsx"

all_sheets = pd.read_excel(excel_file, sheet_name=None)
print(all_sheets)
print("Sheets in the workbook:", list(all_sheets.keys()))

all_sheets["notes"]

{'notes':             sheet                                        description  \
0           ports  Ports the Titanic left from in April 1912. Cod...   
1  board_of_trade  Everyone aboard (passengers and crew, 2,201 pe...   

                                              source  
0  Voyage chronology as reported in standard hist...  
1  British Board of Trade (1912) inquiry report, ...  , 'ports':   code         port   country_today departure_date  stop_number                                               note
0    B      Belfast  United Kingdom     1912-04-02            0  Shipyard; delivery trip only, no passengers in...
1    S  Southampton  United Kingdom     1912-04-10            1                               Maiden voyage begins
2    C    Cherbourg          France     1912-04-10            2                             Evening call in France
3    Q   Queenstown         Ireland     1912-04-11            3  Last stop before the Atlantic crossing (now ca..., 'board_of_trade':   Cl

,sheet,description,source
0,ports,Ports the Titanic left from in April 1912. Cod...,Voyage chronology as reported in standard hist...
1,board_of_trade,"Everyone aboard (passengers and crew, 2,201 pe...","British Board of Trade (1912) inquiry report, ..."


In [30]:
# Read a single sheet by name.
ports = pd.read_excel(excel_file, sheet_name="ports")
ports

,code,port,country_today,departure_date,stop_number,note
0,B,Belfast,United Kingdom,1912-04-02,0,"Shipyard; delivery trip only, no passengers in..."
1,S,Southampton,United Kingdom,1912-04-10,1,Maiden voyage begins
2,C,Cherbourg,France,1912-04-10,2,Evening call in France
3,Q,Queenstown,Ireland,1912-04-11,3,Last stop before the Atlantic crossing (now ca...


In [31]:
board_of_trade = pd.read_excel(excel_file, sheet_name="board_of_trade")
board_of_trade

,Class,Adult_Male_Survived,Adult_Male_Died,Adult_Female_Survived,Adult_Female_Died,Child_Male_Survived,Child_Male_Died,Child_Female_Survived,Child_Female_Died
0,1st,57,118,140,4,5,0,1,0
1,2nd,14,154,80,13,11,0,13,0
2,3rd,75,387,76,89,13,35,14,17
3,Crew,192,670,20,3,0,0,0,0


Look carefully at `board_of_trade`.
A column name such as `Adult_Male_Survived` packs three pieces of information into one label.
That layout is easy for a person to read but awkward to analyze; we will fix it in Sections 10 and 11.

In [32]:
# Write a small subset to a new folder, then read it back to confirm the round trip.
output_dir = Path("/content/isa383/output")
output_dir.mkdir(parents=True, exist_ok=True)

first_five = titanic.head()
first_five.to_csv(output_dir / "first_five_passengers.csv", index=False)

pd.read_csv(output_dir / "first_five_passengers.csv")

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


`index=False` stops pandas from writing the row labels (0 to 4) as an extra, unnamed column.
Forgetting it is the reason many CSV files contain a mysterious `Unnamed: 0` column.

> 💭 **Quick thought question:** With `sheet_name=None`, `pd.read_excel()` returned a dictionary.
Why is a dictionary a natural container for the sheets of a workbook?

### 🧩 Guided practice: Read selectively

Read `titanic.csv` again, keeping only `PassengerId`, `Name`, `Pclass`, `Fare`, and `Embarked`, with `PassengerId` as the index.
Then read only the `ports` sheet from the workbook.
Display the first three rows of each.

In [33]:
# TODO: Read the selected columns with PassengerId as the index.
df = pd.read_csv(DATA_DIR / "titanic.csv",
                 usecols=["PassengerId", "Name", "Pclass", "Fare", "Embarked"],
                 index_col="PassengerId")
display(df.head(3))

# TODO: Read only the ports sheet.
df2 = pd.read_excel(excel_file, sheet_name="ports")
display(df2.head(3))

,Pclass,Name,Fare,Embarked
PassengerId,,,,
1,3,"Braund, Mr. Owen Harris",7.2500,S
2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",71.2833,C
3,3,"Heikkinen, Miss. Laina",7.9250,S


,code,port,country_today,departure_date,stop_number,note
0,B,Belfast,United Kingdom,1912-04-02,0,"Shipyard; delivery trip only, no passengers in..."
1,S,Southampton,United Kingdom,1912-04-10,1,Maiden voyage begins
2,C,Cherbourg,France,1912-04-10,2,Evening call in France


## 6. First look: inspecting the data 🔍

Before any analysis, answer three questions: how big is the table, what type is each column, and where are the gaps?
`info()` answers the last two at once.
`describe()` summarizes numeric columns, and `value_counts()` summarizes categories.
This is also where we start answering the inquiry's questions.

### 🧭 Syntax first: Inspecting a DataFrame

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
df.shape
df.head(n), df.tail(n), df.sample(n, random_state=1)
df.info()
df.describe()                    # numeric columns
df.describe(exclude="number")    # text columns
df["column"].value_counts(normalize=False, dropna=True)
df["column"].unique(), df["column"].nunique()
df.isna().sum()
```

Documentation: [info](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html) · [describe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) · [value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) · [isna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html)

In [34]:
print("Rows, columns:", titanic.shape)
print()
titanic.info()

Rows, columns: (891, 12)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [35]:
# A random sample is often more representative than the first rows of a file.
titanic.sample(5, random_state=383)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
395,396,0,3,"Johansson, Mr. Erik",male,22.0,0,0,350052,7.7958,NaN,S
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
357,358,0,2,"Funk, Miss. Annie Clemmer",female,38.0,0,0,237671,13.0000,NaN,S
272,273,1,2,"Mellinger, Mrs. (Elizabeth Anne Maidment)",female,41.0,0,1,250644,19.5000,NaN,S
537,538,1,1,"LeRoy, Miss. Bertha",female,30.0,0,0,PC 17761,106.4250,NaN,C


In [36]:
titanic.describe().round(2)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.00,891.00,891.00,714.00,891.00,891.00,891.00
mean,446.00,0.38,2.31,29.70,0.52,0.38,32.20
std,257.35,0.49,0.84,14.53,1.10,0.81,49.69
min,1.00,0.00,1.00,0.42,0.00,0.00,0.00
25%,223.50,0.00,2.00,20.12,0.00,0.00,7.91
50%,446.00,0.00,3.00,28.00,0.00,0.00,14.45
75%,668.50,1.00,3.00,38.00,1.00,0.00,31.00
max,891.00,1.00,3.00,80.00,8.00,6.00,512.33


Three things stand out in this summary.
The mean of `Survived` is 0.38: because the column holds 0s and 1s, its mean is the **survival rate**.
`Age` has a count of 714 rather than 891, so 177 ages are missing.
`Fare` ranges from £0 to £512.33: some passengers paid nothing, and someone paid a fortune.

In [37]:
# Text columns: count, number of unique values, most common value (top), and its frequency.
titanic.describe(exclude="number")

,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Dooley, Mr. Patrick",male,347082,G6,S
freq,1,577,7,4,644


There are 891 names but only 681 distinct tickets, so many passengers shared a ticket.
That detail matters later.

In [38]:
# 🕵️ Q1: Who was on board?
print("Passengers by class:")
display(titanic["Pclass"].value_counts().sort_index())
print()
print("Share of passengers by class:")
display(titanic["Pclass"].value_counts(normalize=True).sort_index().round(3))

Passengers by class:


,count
Pclass,
1,216
2,184
3,491



Share of passengers by class:


,proportion
Pclass,
1,0.242
2,0.207
3,0.551


In [39]:
# dropna=False also counts missing values, which value_counts() hides by default.
titanic["Embarked"].value_counts(dropna=False)

,count
Embarked,
S,644
C,168
Q,77
NaN,2


In [40]:
titanic["Survived"]

,Survived
0,0
1,1
2,1
3,1
4,0
...,...
886,0
887,1
888,0
889,1


In [41]:
# 🕵️ Q2: What share of passengers survived?
survivors = titanic["Survived"].sum()
survival_rate = titanic["Survived"].mean()
print(f"{survivors} of {len(titanic)} passengers survived ({survival_rate:.1%}).")

342 of 891 passengers survived (38.4%).


In [42]:
titanic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [43]:
# Where are the gaps?
titanic.isna().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


`Age`, `Cabin`, and `Embarked` have missing values.
We will treat them properly in the *Data Manipulation, Cleaning and Imputation* notebook.
For now, notice that pandas methods such as `mean()` simply skip missing values.

> 💭 **Quick thought question:** More than half of the passengers in this file travelled in third class.
Why does that matter when we report a single overall survival rate?

### 🧩 Guided practice: Inspecting categories

What share of passengers were female?
How many distinct values does `Cabin` have, and how many passengers have no cabin recorded?
Use `value_counts(normalize=True)`, `nunique()`, and `isna().sum()`.

In [44]:
# TODO: Share of passengers by sex.
titanic["Sex"].value_counts(normalize=True).round(4)*100

# TODO: Number of distinct cabins and number of missing cabins.
titanic["Cabin"].value_counts(dropna=False)

titanic.isna().sum()

titanic["Cabin"].nunique()

147

## 7. Selecting columns (and rows) 🎯

Most analyses use only a few columns.
Single brackets with one name return a **Series**; double brackets (a list of names) return a **DataFrame**.
`loc` selects by **labels** (row labels and column names), while `iloc` selects by **integer positions**, just like slicing a list.

### 🧭 Syntax first: Selecting columns and rows

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
df["column"]                          # one column -> Series
df[["column_a", "column_b"]]          # several columns -> DataFrame
df.loc[row_labels, column_labels]     # by label; slices INCLUDE the end
df.iloc[row_positions, col_positions] # by position; slices EXCLUDE the end
df.select_dtypes(include="number")
df.drop(columns=["column"])
df.rename(columns={"old_name": "new_name"})
df["new_column"] = expression
```

Documentation: [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [iloc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html) · [select_dtypes](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.select_dtypes.html) · [drop](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop.html) · [rename](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html)

In [45]:
names = titanic["Name"]                                # one column -> Series
people = titanic[["Name", "Sex", "Age", "Survived"]]   # list of columns -> DataFrame

print(type(names))
print(type(people))
people.head()

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>


,Name,Sex,Age,Survived
0,"Braund, Mr. Owen Harris",male,22.0,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1
2,"Heikkinen, Miss. Laina",female,26.0,1
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1
4,"Allen, Mr. William Henry",male,35.0,0


In [46]:
# loc: rows labelled 1 through 4 (the end label IS included) and two named columns.
titanic.loc[1:4, ["Name", "Fare"]]

,Name,Fare
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",71.2833
2,"Heikkinen, Miss. Laina",7.9250
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",53.1000
4,"Allen, Mr. William Henry",8.0500


In [47]:
# iloc: the first five rows and first three columns by position (the end is NOT included).
titanic.iloc[0:5, 0:3]

,PassengerId,Survived,Pclass
0,1,0,3
1,2,1,1
2,3,1,3
3,4,1,1
4,5,0,3


`loc[0:4]` returns five rows, and `iloc[0:5]` also returns five rows.
The difference in the end point is a common source of off-by-one mistakes.

In [48]:
# Keep only numeric columns.
titanic.select_dtypes(include="number").head(3)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
0,1,0,3,22.0,1,0,7.2500
1,2,1,1,38.0,1,0,71.2833
2,3,1,3,26.0,0,0,7.9250


In [49]:
# A list comprehension (from the Iterations notebook) can choose columns by a rule:
# here, the columns that contain at least one missing value.
columns_with_gaps = [column for column in titanic.columns if titanic[column].isna().any()]
columns_with_gaps

['Age', 'Cabin', 'Embarked']

In [50]:
# rename() and drop() return a NEW DataFrame; the original is unchanged.
readable = titanic.rename(
    columns={"Pclass": "ticket_class", "SibSp": "siblings_spouses", "Parch": "parents_children"}
).drop(columns=["Ticket", "Cabin"])

readable.head(3)

,PassengerId,Survived,ticket_class,Name,Sex,Age,siblings_spouses,parents_children,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S


In [51]:
# Create a new column from existing ones: the passenger plus relatives travelling with them.
titanic["family_size"] = titanic["SibSp"] + titanic["Parch"] + 1

titanic[["Name", "SibSp", "Parch", "family_size"]].head()

,Name,SibSp,Parch,family_size
0,"Braund, Mr. Owen Harris",1,0,2
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0,2
2,"Heikkinen, Miss. Laina",0,0,1
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0,2
4,"Allen, Mr. William Henry",0,0,1


In [52]:
titanic["family_size"].describe()

,family_size
count,891.000000
mean,1.904602
std,1.613459
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,11.000000


In [53]:
titanic["family_size"].value_counts(ascending=False)

,count
family_size,
1,537
2,161
3,102
4,29
6,22
5,15
7,12
11,7
8,6


> 💭 **Quick thought question:** What is the difference between `titanic["Age"]` and `titanic[["Age"]]`?
When would you want each one?

### 🧩 Guided practice: loc and iloc

Use `loc` to select the rows labelled 10 to 14 and the columns `Name`, `Pclass`, and `Fare`.
Then use `iloc` to select the last three rows and the last two columns.

*Hint:* negative positions count from the end, just as in lists.

In [54]:
# TODO: loc selection.

# TODO: iloc selection.

## 8. Filtering rows 🔎

A filter asks a question of every row at once.
A comparison such as `titanic["Age"] < 16` returns a Boolean Series, often called a **mask**, with `True` for the rows that match.
Putting the mask inside square brackets keeps only the `True` rows.

Combine conditions with `&` (and), `|` (or), and `~` (not), and wrap each condition in parentheses.
Python's `and` and `or` keywords do not work on whole columns.

### 🧭 Syntax first: Filtering rows

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
mask = df["column"] > value
df[mask]
df[(df["a"] == x) & (df["b"] < y)]          # and
df[(df["a"] == x) | (df["b"] < y)]          # or
df[~mask]                                   # not
df[df["column"].isin(["v1", "v2"])]
df[df["column"].between(low, high)]         # inclusive on both ends
df[df["text"].str.contains("pattern", regex=False)]
df.query("a == 'x' and b < 10")
df.loc[mask, ["column_a", "column_b"]]      # filter rows AND select columns
```

Documentation: [Boolean indexing](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing) · [isin](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html) · [between](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html) · [str.contains](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html) · [query](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html)

In [55]:
is_child = titanic["Age"] < 16

print(is_child.head())
print("Number of children (under 16):", is_child.sum())

0    False
1    False
2    False
3    False
4    False
Name: Age, dtype: bool
Number of children (under 16): 83


In [56]:
children = titanic[is_child]
children[["Name", "Age", "Pclass", "Survived"]].head()

,Name,Age,Pclass,Survived
7,"Palsson, Master. Gosta Leonard",2.0,3,0
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.0,2,1
10,"Sandstrom, Miss. Marguerite Rut",4.0,3,1
14,"Vestrom, Miss. Hulda Amanda Adolfina",14.0,3,0
16,"Rice, Master. Eugene",2.0,3,0


In [58]:
titanic.query("Age<16")

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,family_size
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.00,3,1,349909,21.0750,NaN,S,5
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.00,1,0,237736,30.0708,NaN,C,2
10,11,1,3,"Sandstrom, Miss. Marguerite Rut",female,4.00,1,1,PP 9549,16.7000,G6,S,3
14,15,0,3,"Vestrom, Miss. Hulda Amanda Adolfina",female,14.00,0,0,350406,7.8542,NaN,S,1
16,17,0,3,"Rice, Master. Eugene",male,2.00,4,1,382652,29.1250,NaN,Q,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
831,832,1,2,"Richards, Master. George Sibley",male,0.83,1,1,29106,18.7500,NaN,S,3
850,851,0,3,"Andersson, Master. Sigvard Harald Elias",male,4.00,4,2,347082,31.2750,NaN,S,7
852,853,0,3,"Boulos, Miss. Nourelain",female,9.00,1,1,2678,15.2458,NaN,C,3
869,870,1,3,"Johnson, Master. Harold Theodor",male,4.00,1,1,347742,11.1333,NaN,S,3


In [59]:
# 🕵️ Q3: Was "women and children first" followed?
women = titanic[titanic["Sex"] == "female"]
children = titanic[titanic["Age"] < 16]
adult_men = titanic[(titanic["Sex"] == "male") & (titanic["Age"] >= 16)]

# Using .query()
women = titanic.query("Sex == 'female'")
children = titanic.query("Age < 16")
adult_men = titanic.query("Sex == 'male' and Age >= 16")

print(f"Women:     {women['Survived'].mean():.1%} survived (n = {len(women)})")
print(f"Children:  {children['Survived'].mean():.1%} survived (n = {len(children)})")
print(f"Adult men: {adult_men['Survived'].mean():.1%} survived (n = {len(adult_men)})")

Women:     74.2% survived (n = 314)
Children:  59.0% survived (n = 83)
Adult men: 17.4% survived (n = 413)


The pattern is stark: about three in four women survived, compared with fewer than one in five adult men.
Children did better than men but much worse than women, which raises a question the Advanced notebook will answer: did every child get the same chance?

In [60]:
# isin(): passengers who boarded in France or Ireland.
continental = titanic[titanic["Embarked"].isin(["C", "Q"])]
display(continental)
continental["Embarked"].value_counts()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,family_size
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q,1
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C,2
16,17,0,3,"Rice, Master. Eugene",male,2.0,4,1,382652,29.1250,NaN,Q,6
19,20,1,3,"Masselmani, Mrs. Fatima",female,NaN,0,0,2649,7.2250,NaN,C,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
875,876,1,3,"Najib, Miss. Adele Kiamie ""Jane""",female,15.0,0,0,2667,7.2250,NaN,C,1
879,880,1,1,"Potter, Mrs. Thomas Jr (Lily Alexenia Wilson)",female,56.0,0,1,11767,83.1583,C50,C,2
885,886,0,3,"Rice, Mrs. William (Margaret Norton)",female,39.0,0,5,382652,29.1250,NaN,Q,6
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,1


,count
Embarked,
C,168
Q,77


In [61]:
# between(): passengers in their twenties (20 and 29 are both included).
twenties = titanic[titanic["Age"].between(20, 29)]
print("Passengers in their twenties:", len(twenties))

Passengers in their twenties: 220


In [65]:
titanic.query("Age <= 29 and Age >= 20")

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,family_size
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S,3
12,13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.0500,NaN,S,1
23,24,1,1,"Sloper, Mr. William Thompson",male,28.0,0,0,113788,35.5000,A6,S,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
882,883,0,3,"Dahlberg, Miss. Gerda Ulrika",female,22.0,0,0,7552,10.5167,NaN,S,1
883,884,0,2,"Banfield, Mr. Frederick James",male,28.0,0,0,C.A./SOTON 34068,10.5000,NaN,S,1
884,885,0,3,"Sutehall, Mr. Henry Jr",male,25.0,0,0,SOTON/OQ 392076,7.0500,NaN,S,1
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,1


In [62]:
# str.contains(): "Master." was the title used for boys.
boys = titanic[titanic["Name"].str.contains("Master.", regex=False)]
print("Boys with the title Master:", len(boys), "| survival rate:", round(boys["Survived"].mean(), 3))

clergy = titanic[titanic["Name"].str.contains("Rev.", regex=False)]
clergy[["Name", "Age", "Pclass", "Survived"]]

Boys with the title Master: 40 | survival rate: 0.575


,Name,Age,Pclass,Survived
149,"Byles, Rev. Thomas Roussel Davids",42.0,2,0
150,"Bateman, Rev. Robert James",51.0,2,0
249,"Carter, Rev. Ernest Courtenay",54.0,2,0
626,"Kirkland, Rev. Charles Leonard",57.0,2,0
848,"Harper, Rev. John",28.0,2,0
886,"Montvila, Rev. Juozas",27.0,2,0


All six clergymen in the data died.

In [63]:
# ~ means "not": passengers WITH a recorded cabin.
with_cabin = titanic[~titanic["Cabin"].isna()]
print("Passengers with a recorded cabin:", len(with_cabin))

Passengers with a recorded cabin: 204


In [64]:
# query() reads almost like a sentence: first-class women who did NOT survive.
titanic.query("Pclass == 1 and Sex == 'female' and Survived == 0")[["Name", "Age", "Fare"]]

,Name,Age,Fare
177,"Isham, Miss. Ann Elizabeth",50.0,28.7125
297,"Allison, Miss. Helen Loraine",2.0,151.5500
498,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",25.0,151.5500


Only three first-class women in the data died.
Two of them were Bess Allison and her two-year-old daughter Loraine.
According to most accounts, they stayed on board looking for the family's baby son, who had already been carried to a lifeboat by his nurse.

In [68]:
titanic[titanic["Fare"] == 0]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,family_size
179,180,0,3,"Leonard, Mr. Lionel",male,36.0,0,0,LINE,0.0,NaN,S,1
263,264,0,1,"Harrison, Mr. William",male,40.0,0,0,112059,0.0,B94,S,1
271,272,1,3,"Tornquist, Mr. William Henry",male,25.0,0,0,LINE,0.0,NaN,S,1
277,278,0,2,"Parkes, Mr. Francis ""Frank""",male,NaN,0,0,239853,0.0,NaN,S,1
302,303,0,3,"Johnson, Mr. William Cahoone Jr",male,19.0,0,0,LINE,0.0,NaN,S,1
413,414,0,2,"Cunningham, Mr. Alfred Fleming",male,NaN,0,0,239853,0.0,NaN,S,1
466,467,0,2,"Campbell, Mr. William",male,NaN,0,0,239853,0.0,NaN,S,1
481,482,0,2,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0,NaN,S,1
597,598,0,3,"Johnson, Mr. Alfred",male,49.0,0,0,LINE,0.0,NaN,S,1
633,634,0,1,"Parr, Mr. William Henry Marsh",male,NaN,0,0,112052,0.0,NaN,S,1


In [ ]:
# 🕵️ Q4 (part 1): Who travelled for free?
titanic.loc[titanic["Fare"] == 0, ["Name", "Pclass", "Ticket", "Survived"]]

,Name,Pclass,Ticket,Survived
179,"Leonard, Mr. Lionel",3,LINE,0
263,"Harrison, Mr. William",1,112059,0
271,"Tornquist, Mr. William Henry",3,LINE,1
277,"Parkes, Mr. Francis ""Frank""",2,239853,0
302,"Johnson, Mr. William Cahoone Jr",3,LINE,0
413,"Cunningham, Mr. Alfred Fleming",2,239853,0
466,"Campbell, Mr. William",2,239853,0
481,"Frost, Mr. Anthony Wood ""Archie""",2,239854,0
597,"Johnson, Mr. Alfred",3,LINE,0
633,"Parr, Mr. William Henry Marsh",1,112052,0


A fare of £0 looks like a data error, but it is not.
Thomas Andrews (ticket 112050), the Harland & Wolff naval architect who led the ship's design, sailed with a small "guarantee group" of shipyard staff to observe the maiden voyage.
Most of the other £0 fares belong to employees of the White Star Line or other shipping lines, including the tickets marked `LINE`.
Only one of these 15 passengers survived, and Andrews was not among them.

> 💭 **Quick thought question:** `titanic[titanic["Age"] < 16]` silently leaves out the 177 passengers whose age is missing, because a comparison with `NaN` is always `False`.
How might that affect our answer to Q3?

### 🧩 Guided practice: Children by class

Filter the children (under 16) who travelled in **third** class and compute their survival rate.
Then do the same for children in first **or** second class.
What does the comparison suggest about "women and children first"?

In [77]:
# TODO: Third-class children. - only this for now.
df = titanic[(titanic['Age']<16) & (titanic['Pclass']==3)]  [['Age', 'Pclass', 'Survived']]
df = titanic.query("Age < 16 and Pclass == 3") [['Age', 'Pclass', 'Survived']]
df.head()

df['Survived'].mean()


# TODO: First- or second-class children.

np.float64(0.43103448275862066)

## 9. Sorting ↕️

Sorting answers "who is at the top or the bottom?"
`sort_values()` sorts by one or more columns, and `ascending` can be a list with one entry per column.
`nlargest()` and `nsmallest()` are shortcuts for a top-*n* list.
Like most pandas methods, sorting returns a new DataFrame; the original keeps its order unless you assign the result.

### 🧭 Syntax first: Sorting

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
df.sort_values(by="column", ascending=True, na_position="last")
df.sort_values(by=["column_a", "column_b"], ascending=[True, False])
df.nlargest(n, "column")
df.nsmallest(n, "column")
df.sort_index()
```

Documentation: [sort_values](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html) · [nlargest](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nlargest.html) · [sort_index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_index.html)

In [78]:
# 🕵️ Q4 (part 2): Who paid the most?
(
    titanic.sort_values("Fare", ascending=False)
    [["Name", "Pclass", "Ticket", "Fare", "Survived"]]
    .head(6)
)

,Name,Pclass,Ticket,Fare,Survived
679,"Cardeza, Mr. Thomas Drake Martinez",1,PC 17755,512.3292,1
258,"Ward, Miss. Anna",1,PC 17755,512.3292,1
737,"Lesurer, Mr. Gustave J",1,PC 17755,512.3292,1
88,"Fortune, Miss. Mabel Helen",1,19950,263.0000,1
438,"Fortune, Mr. Mark",1,19950,263.0000,0
341,"Fortune, Miss. Alice Elizabeth",1,19950,263.0000,1


The top fare, £512.33, was paid on ticket PC 17755 by Thomas Cardeza, a wealthy American travelling with his mother Charlotte, his manservant Gustave Lesurer, and her maid Anna Ward in one of the most luxurious suites on board.
Charlotte Cardeza is not in our 891-passenger subset, but the other three are.
The same £512.33 appears on three rows, yet nobody paid it three times: **the fare belongs to the ticket, not to each passenger.**
Keep that in mind; it is a tidy-data problem we diagnose in the next section.

In [80]:
(titanic
  .sort_values(["Pclass", "Age"], ascending=[True, False])
  [["Name", "Pclass", "Age", "Survived"]]
 .head(8)
)

,Name,Pclass,Age,Survived
630,"Barkworth, Mr. Algernon Henry Wilson",1,80.0,1
96,"Goldschmidt, Mr. George B",1,71.0,0
493,"Artagaveytia, Mr. Ramon",1,71.0,0
745,"Crosby, Capt. Edward Gifford",1,70.0,0
54,"Ostby, Mr. Engelhart Cornelius",1,65.0,0
456,"Millet, Mr. Francis Davis",1,65.0,0
438,"Fortune, Mr. Mark",1,64.0,0
545,"Nicholson, Mr. Arthur Ernest",1,64.0,0


In [79]:
# Sort by two columns: class ascending, then age descending within each class.
(
    titanic.sort_values(["Pclass", "Age"], ascending=[True, False])
    [["Name", "Pclass", "Age", "Survived"]]
    .head(8)
)

,Name,Pclass,Age,Survived
630,"Barkworth, Mr. Algernon Henry Wilson",1,80.0,1
96,"Goldschmidt, Mr. George B",1,71.0,0
493,"Artagaveytia, Mr. Ramon",1,71.0,0
745,"Crosby, Capt. Edward Gifford",1,70.0,0
54,"Ostby, Mr. Engelhart Cornelius",1,65.0,0
456,"Millet, Mr. Francis Davis",1,65.0,0
438,"Fortune, Mr. Mark",1,64.0,0
545,"Nicholson, Mr. Arthur Ernest",1,64.0,0


In [81]:
print("Oldest passengers:")
display(titanic.nlargest(3, "Age")[["Name", "Age", "Pclass", "Survived"]])

print("Youngest passengers:")
display(titanic.nsmallest(3, "Age")[["Name", "Age", "Pclass", "Survived"]])

Oldest passengers:


,Name,Age,Pclass,Survived
630,"Barkworth, Mr. Algernon Henry Wilson",80.0,1,1
851,"Svensson, Mr. Johan",74.0,3,0
96,"Goldschmidt, Mr. George B",71.0,1,0


Youngest passengers:


,Name,Age,Pclass,Survived
803,"Thomas, Master. Assad Alexander",0.42,3,1
755,"Hamalainen, Master. Viljo",0.67,2,1
469,"Baclini, Miss. Helene Barbara",0.75,3,1


The youngest passenger, five-month-old Assad Thomas, survived.
The oldest recorded passenger, Algernon Barkworth, survived too, but *Encyclopedia Titanica* gives his birth year as 1864, which would make him 47, not 80.
Even a famous, heavily studied dataset can contain recording errors, so treat surprising values as questions rather than facts.

In [82]:
# By default, missing values go last. Putting them first makes them easy to inspect.
titanic.sort_values("Age", na_position="first")[["Name", "Pclass", "Age"]].head()

,Name,Pclass,Age
5,"Moran, Mr. James",3,NaN
17,"Williams, Mr. Charles Eugene",2,NaN
19,"Masselmani, Mrs. Fatima",3,NaN
26,"Emir, Mr. Farred Chehab",3,NaN
28,"O'Dwyer, Miss. Ellen ""Nellie""",3,NaN


> 💭 **Quick thought question:** When two passengers have the same fare, in what order does pandas list them?
How could you make the order fully predictable?

### 🧩 Guided practice: The largest families

Sort the passengers by `family_size` (largest first) and then by `Name` (A to Z).
Show the top 12 rows with `Name`, `Pclass`, `family_size`, and `Survived`.
What do you notice about the class and the outcome of the largest families?

In [85]:
# TODO: Sort by family_size (descending) and Name (ascending).

(titanic
  .sort_values(["family_size", "Name"], ascending=[False, True])
  .head(12)
  [["Name", "Pclass", "family_size", "Survived"]]
)


,Name,Pclass,family_size,Survived
159,"Sage, Master. Thomas Henry",3,11,0
180,"Sage, Miss. Constance Gladys",3,11,0
863,"Sage, Miss. Dorothy Edith ""Dolly""",3,11,0
792,"Sage, Miss. Stella Anna",3,11,0
846,"Sage, Mr. Douglas Bullen",3,11,0
201,"Sage, Mr. Frederick",3,11,0
324,"Sage, Mr. George John Jr",3,11,0
480,"Goodwin, Master. Harold Victor",3,8,0
386,"Goodwin, Master. Sidney Leonard",3,8,0
59,"Goodwin, Master. William Frederick",3,8,0


## 10. Tidy data principles 🧹

Hadley Wickham's paper *Tidy Data* (*Journal of Statistical Software*, 2014) gave a name to a table layout that makes analysis easy.
A dataset is **tidy** when:

1. Each **variable** forms a column.
2. Each **observation** forms a row.
3. Each type of **observational unit** forms its own table.

A consequence is that each value sits in its own cell.
Tidy data is not about looking neat; it is about predictability.
When every table follows the same rules, the same pandas tools (filtering, `groupby()`, `merge()`, and plotting) work on all of them without special handling.

Three problems break the rules most often:

| Problem | *Titanic* example | Typical fix |
|---|---|---|
| Column headers are values, not variable names | `board_of_trade` has columns like `Adult_Male_Survived` | `melt()` to a long layout |
| Several variables stored in one column | `Name` holds surname, title, and given names | `str.split()` or `str.extract()` |
| Two kinds of observational unit in one table | `Fare` describes a ticket but is repeated for every passenger on that ticket | Separate tables, joined with `merge()` when needed |

Is `titanic.csv` tidy?
Mostly yes: each row is one passenger, and each column is one variable.
The two exceptions below are worth seeing.

In [ ]:
# Problem: several variables in one column.
titanic["Name"].head()

0                              Braund, Mr. Owen Harris
1    Cumings, Mrs. John Bradley (Florence Briggs Th...
2                               Heikkinen, Miss. Laina
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                             Allen, Mr. William Henry
Name: Name, dtype: object

In [86]:
# Split once at ", " to separate the surname; the title is everything before the first ".".
name_parts = titanic["Name"].str.split(", ", n=1, expand=True)
name_parts.columns = ["surname", "rest"]
name_parts["title"] = name_parts["rest"].str.split(".", n=1).str[0]

name_parts.head()

,surname,rest,title
0,Braund,Mr. Owen Harris,Mr
1,Cumings,Mrs. John Bradley (Florence Briggs Thayer),Mrs
2,Heikkinen,Miss. Laina,Miss
3,Futrelle,Mrs. Jacques Heath (Lily May Peel),Mrs
4,Allen,Mr. William Henry,Mr


In [87]:
name_parts["title"].value_counts().head(8)

,count
title,
Mr,517
Miss,182
Mrs,125
Master,40
Dr,7
Rev,6
Col,2
Mlle,2


Now `title` is a variable in its own right.
"Master" identifies boys, which will help us estimate missing ages in the cleaning notebook.

In [88]:
# Problem: two observational units (passengers and tickets) mixed in one table.
titanic[titanic["Ticket"] == "PC 17755"][["Name", "Pclass", "Ticket", "Fare"]]

,Name,Pclass,Ticket,Fare
258,"Ward, Miss. Anna",1,PC 17755,512.3292
679,"Cardeza, Mr. Thomas Drake Martinez",1,PC 17755,512.3292
737,"Lesurer, Mr. Gustave J",1,PC 17755,512.3292


In [89]:
ticket_counts = titanic["Ticket"].value_counts()
print("Tickets shared by two or more passengers:", (ticket_counts > 1).sum())
print("Passengers travelling on a shared ticket:", ticket_counts[ticket_counts > 1].sum())

Tickets shared by two or more passengers: 134
Passengers travelling on a shared ticket: 344


In [90]:
# A tidier design: one table per observational unit.
tickets = titanic[["Ticket", "Pclass", "Fare"]].drop_duplicates(subset="Ticket")
passengers_only = titanic.drop(columns=["Fare"])

print("Tickets table:   ", tickets.shape, "-> one row per ticket")
print("Passengers table:", passengers_only.shape, "-> one row per passenger")
tickets.head()

Tickets table:    (681, 3) -> one row per ticket
Passengers table: (891, 12) -> one row per passenger


,Ticket,Pclass,Fare
0,A/5 21171,3,7.2500
1,PC 17599,1,71.2833
2,STON/O2. 3101282,3,7.9250
3,113803,1,53.1000
4,373450,3,8.0500


The two tables share the `Ticket` column, which acts as a **key**.
In the Advanced notebook, `merge()` uses keys like this to bring related tables back together, and `groupby()` lets us compute the fare per person.

> 💭 **Quick thought question:** In `board_of_trade`, what is one observation?
Which variables are hiding in the column names?

## 11. Long versus wide data ↔️

The same information can be laid out in two shapes.
In a **wide** table, the values of one variable are spread across several columns, for example one column per outcome.
In a **long** table, those columns are stacked: one column says *which* measurement it is, and another holds the *value*.

**Wide** (one row per class):

| Class | Survived | Died |
|---|---|---|
| 1st | 203 | 122 |
| 2nd | 118 | 167 |

**Long** (one row per class and outcome):

| Class | outcome | people |
|---|---|---|
| 1st | Survived | 203 |
| 1st | Died | 122 |
| 2nd | Survived | 118 |
| 2nd | Died | 167 |

Wide tables are compact and easy for people to read in a report.
Long tables are what `groupby()`, `merge()`, and seaborn charts expect, and they are usually the tidy form.
`melt()` turns wide into long, and `pivot()` turns long into wide.

### 🧭 Syntax first: Melting and pivoting

Read the pattern before running the example.
The names in the pattern are placeholders.

```python
df.melt(
    id_vars=["id_column"],            # columns that identify each row; kept as they are
    value_vars=["col_a", "col_b"],    # columns to stack (default: all the others)
    var_name="variable",              # name for the column of former headers
    value_name="value",               # name for the column of values
)

df.pivot(index="row_labels", columns="column_labels", values="value")
```

Documentation: [melt](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.melt.html) · [pivot](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html)

In [ ]:
# The Board of Trade table: wide, with three variables packed into each column name.
board_of_trade

,Class,Adult_Male_Survived,Adult_Male_Died,Adult_Female_Survived,Adult_Female_Died,Child_Male_Survived,Child_Male_Died,Child_Female_Survived,Child_Female_Died
0,1st,57,118,140,4,5,0,1,0
1,2nd,14,154,80,13,11,0,13,0
2,3rd,75,387,76,89,13,35,14,17
3,Crew,192,670,20,3,0,0,0,0


In [91]:
# Wide -> long: keep Class as the identifier and stack the other eight columns.
board_long = board_of_trade.melt(
    id_vars="Class",
    var_name="group",
    value_name="people",
)

print("Wide shape:", board_of_trade.shape, "-> long shape:", board_long.shape)
board_long.head(6)

Wide shape: (4, 9) -> long shape: (32, 3)


,Class,group,people
0,1st,Adult_Male_Survived,57
1,2nd,Adult_Male_Survived,14
2,3rd,Adult_Male_Survived,75
3,Crew,Adult_Male_Survived,192
4,1st,Adult_Male_Died,118
5,2nd,Adult_Male_Died,154


In [92]:
# Split the packed column into three variables, then drop the original.
board_long[["age_group", "sex", "outcome"]] = board_long["group"].str.split("_", expand=True)
board_tidy = board_long.drop(columns="group")[["Class", "age_group", "sex", "outcome", "people"]]

board_tidy.head(8)

,Class,age_group,sex,outcome,people
0,1st,Adult,Male,Survived,57
1,2nd,Adult,Male,Survived,14
2,3rd,Adult,Male,Survived,75
3,Crew,Adult,Male,Survived,192
4,1st,Adult,Male,Died,118
5,2nd,Adult,Male,Died,154
6,3rd,Adult,Male,Died,387
7,Crew,Adult,Male,Died,670


Now each row is one observation, the number of people in one class, age group, sex, and outcome combination, and each column is one variable.
Questions that were awkward on the wide table become short filters.

In [93]:
# 🕵️ Q5: How do our 891 passengers compare with everyone aboard?
total_aboard = board_tidy["people"].sum()
total_survived = board_tidy.loc[board_tidy["outcome"] == "Survived", "people"].sum()

crew_aboard = board_tidy.loc[board_tidy["Class"] == "Crew", "people"].sum()
crew_survived = board_tidy.loc[
    (board_tidy["Class"] == "Crew") & (board_tidy["outcome"] == "Survived"), "people"
].sum()

print(f"Everyone aboard: {total_aboard:,} people, {total_survived} survived ({total_survived / total_aboard:.1%})")
print(f"Crew:            {crew_aboard} people, {crew_survived} survived ({crew_survived / crew_aboard:.1%})")
print(f"Our passengers:  {len(titanic)} people, {titanic['Survived'].sum()} survived ({titanic['Survived'].mean():.1%})")

Everyone aboard: 2,201 people, 711 survived (32.3%)
Crew:            885 people, 212 survived (24.0%)
Our passengers:  891 people, 342 survived (38.4%)


About 40% of the people aboard were crew, and fewer than one in four of them survived.
That is why the survival rate for everyone aboard (32.3%) is lower than the rate for our passenger file (38.4%).
A dataset that leaves out a large group can tell a different story from the full population.

In [94]:
# Long -> wide: one row per class, one column per outcome, for adult men only.
adult_men_long = board_tidy[(board_tidy["age_group"] == "Adult") & (board_tidy["sex"] == "Male")]
adult_men_wide = adult_men_long.pivot(index="Class", columns="outcome", values="people")

adult_men_wide["survival_rate"] = (
    adult_men_wide["Survived"] / (adult_men_wide["Survived"] + adult_men_wide["Died"])
).round(3)
adult_men_wide

outcome,Died,Survived,survival_rate
Class,,,
1st,118,57,0.326
2nd,154,14,0.083
3rd,387,75,0.162
Crew,670,192,0.223


The wide layout is the right choice here because we want to compare two outcomes side by side and compute a rate across them.
Notice the surprise: adult men in **second** class had the lowest survival rate of any group, below third-class men and the crew.
`pivot()` needs each index-column pair to appear only once; when there are duplicates, `pivot_table()` (Advanced notebook) aggregates them instead.

> 💭 **Quick thought question:** Which layout would you paste into an email to a manager?
Which would you hand to pandas or a plotting library?
Why are they different?

### 🧩 Guided practice: Children in the Board of Trade data

Filter `board_tidy` to girls (`age_group == "Child"` and `sex == "Female"`) and pivot the result so that each row is a class and the columns are `Died` and `Survived`.
Repeat for boys.
In which classes did children die?

In [ ]:
# TODO: Filter to children, then reshape to a wide table.

## 12. Bringing it together: the inquiry's first findings 📝

Analysts rarely stop at a table; they write down what it means.
The cell below collects the numbers behind our first five answers, and the memo after it interprets them.

In [ ]:
women = titanic[titanic["Sex"] == "female"]
men = titanic[titanic["Sex"] == "male"]
first_class = titanic[titanic["Pclass"] == 1]
third_class = titanic[titanic["Pclass"] == 3]

findings = pd.DataFrame(
    {
        "group": ["All passengers", "Women", "Men", "First class", "Third class", "Everyone aboard (incl. crew)"],
        "people": [len(titanic), len(women), len(men), len(first_class), len(third_class), total_aboard],
        "survival_rate": [
            titanic["Survived"].mean(),
            women["Survived"].mean(),
            men["Survived"].mean(),
            first_class["Survived"].mean(),
            third_class["Survived"].mean(),
            total_survived / total_aboard,
        ],
    }
)
findings["survival_rate"] = findings["survival_rate"].round(3)
findings.sort_values("survival_rate", ascending=False)

,group,people,survival_rate
1,Women,314,0.742
3,First class,216,0.630
0,All passengers,891,0.384
5,Everyone aboard (incl. crew),2201,0.323
4,Third class,491,0.242
2,Men,577,0.189


**Memo to the inquiry (draft).**
Of the 891 passengers in our records, 342 (38.4%) survived.
Women survived at roughly four times the rate of men (74.2% versus 18.9%), which is consistent with a "women and children first" policy, although children survived at only 59%.
First-class passengers survived at 63.0%, compared with 24.2% in third class.
Fares ranged from nothing, for shipyard and shipping-line staff, to £512.33 for a single first-class suite.
Including the crew, only 32.3% of the 2,201 people aboard survived.

**Limitations.**
These records cover about two-thirds of the passengers and none of the crew, 177 ages are missing, and a comparison of two groups does not tell us *why* they differ.
The Advanced notebook asks whether class and sex worked together, and whether the port of embarkation mattered.

## Comprehensive exercises 📝

Complete the exercises below.
For each one, display a small result and write one or two sentences explaining what it means.
Readable code in a few clear steps is better than a clever one-liner.

### Exercise 1: A selective first look

Read `titanic.csv` with `PassengerId` as the index and only the columns `Name`, `Pclass`, `Sex`, `Age`, `Fare`, and `Survived`.
Report the shape, the dtypes, the number of missing ages, and the median fare.
Then use `value_counts()` to report how many passengers of each sex travelled in each class (*hint:* pass a list of two columns to `value_counts()` on the DataFrame).

In [ ]:
# Write your solution here.

### Exercise 2: Women and children first, class by class

Using filters only (no `groupby()` yet), compute the survival rate for each of these groups, along with the number of people in it:

1. First-class women
2. Third-class women
3. Third-class children (under 16)
4. Second-class men aged 16 or over

Put the results in a small DataFrame sorted from the highest rate to the lowest.
Was "women and children first" applied equally across classes?

In [ ]:
# Write your solution here.

### Exercise 3: The fare puzzle

Some third-class passengers paid more than some first-class passengers.
Find every third-class passenger with a fare above £30, show `Name`, `Ticket`, `Fare`, and `family_size`, and sort by `Fare` (highest first) and then by `Name`.
Use what you learned in Section 10 to explain the puzzle in two sentences.

In [ ]:
# Write your solution here.

### Exercise 4: Reshaping the Board of Trade report

Starting from `board_of_trade` (the wide sheet), produce a tidy long table without looking back at Section 11.
Then answer, using filters and `sum()`:

1. How many women were aboard, and what share of them survived?
2. How many men were aboard, and what share of them survived?

Finally, reshape the women-only rows into a wide table with one row per class and one column per outcome, and add a `survival_rate` column.

In [ ]:
# Write your solution here.

### Exercise 5: Your own inquiry question

Write one question about the passengers that a 1912 investigator, a historian, or a modern cruise-line safety officer might ask.
Answer it using at least one filter, one sort, and one column selection.
Write a three-sentence answer: the finding, the evidence (numbers), and one limitation of this dataset.

In [ ]:
# Write your solution here.

## Summary ✅

- A **Series** is a labeled column; a **DataFrame** is a table of Series that share an index.
- `read_csv()` and `read_excel()` load files; parameters such as `usecols`, `index_col`, `nrows`, and `sheet_name` save work later. Always inspect with `shape`, `info()`, `describe()`, and `value_counts()`.
- Select columns with `[]`, rows and columns with `loc` (labels) and `iloc` (positions), filter with Boolean masks, `isin()`, `between()`, `str.contains()`, or `query()`, and sort with `sort_values()`.
- **Tidy data**: each variable is a column, each observation is a row, and each type of observational unit is a table.
- **Wide** layouts suit people; **long** layouts suit analysis. `melt()` goes wide to long, and `pivot()` goes long to wide.

**Next:** *Pandas for Data Wrangling (Advanced)* combines tables with `concat()`, `merge()`, and `join()`, and summarizes them with `groupby()` and aggregations to answer Q6 and Q7.